# Museum Analysis

This notebook connects to the museum database and conducts simple analysis of the data.

## Imports

In [45]:
from extract import get_s3_client, list_objects, download_objects, extract_csv, choose_bucket, combine_csv_files
from psycopg2 import Error, connect
from os import environ as ENV, _Environ, mkdir, listdir, path
from os.path import exists, join, dirname

import pandas as pd

from dotenv import load_dotenv

## Setup

In [46]:
load_dotenv()


def get_db_connection():
    """Returns database connection"""
    try:
        connection = connect(
            dbname=ENV["DB_NAME"],
            user=ENV["DB_USERNAME"],
            password=ENV['DB_PASSWORD'],
            host=ENV['DB_HOST'],
            port=ENV["DB_PORT"]
        )
        return connection
    except Error as e:
        print(f"Error connecting to database: {e}")
        return None

In [47]:
conn = get_db_connection()

# Task 3

## Querying the database

### What exhibition is most frequently visited?

In [54]:
conn.rollback()

In [55]:
with conn.cursor() as cur:
    query = """
    WITH visits AS (
        SELECT exhibition_id
        FROM request_interaction
        UNION ALL
        SELECT exhibition_id
        FROM rating_interaction
    )
    SELECT
        e.exhibition_id, 
        e.exhibition_name,
        COUNT(*) AS visit_count
    FROM visits v
    JOIN exhibition e
        USING(exhibition_id)
    GROUP BY e.exhibition_id
    ORDER BY visit_count DESC
    LIMIT 5
;

"""
    cur.execute(query)
    result = cur.fetchall()

result

[(3, 'Cetacean Sensations', 501),
 (4, 'Our Polluted World', 492),
 (1, 'Adaptation', 435),
 (2, 'The Crenshaw Collector', 399),
 (0, 'Measureless to Man', 310)]

We can see here that with 501 total interactions (both requests and ratings), Cetacean Sensations is the most visited exhibit.

### What hour of the day has the most ratings?

In [56]:
with conn.cursor() as cur:
    query = """
    SELECT
        EXTRACT(HOUR FROM event_at) as hour_of_day,
        COUNT(*) AS rating_count
    FROM rating_interaction
    GROUP BY hour_of_day
    ORDER BY rating_count DESC
    LIMIT 5
;

"""
    cur.execute(query)
    result = cur.fetchall()

result

[(Decimal('10'), 232),
 (Decimal('13'), 228),
 (Decimal('17'), 225),
 (Decimal('14'), 224),
 (Decimal('9'), 221)]

With 232 visits we can see that 10 AM is the most popular time of day based on the ratings received at this hour.

### What exhibition has the most emergencies?

In [57]:
with conn.cursor() as cur:
    query = """
    SELECT
        e.exhibition_id,
        e.exhibition_name,
        COUNT(*) AS emergency_count
    FROM request_interaction ri
    JOIN exhibition e
        USING(exhibition_id)
    WHERE ri.request_id = 1
    GROUP BY e.exhibition_id, e.exhibition_name
    ORDER BY emergency_count DESC
    LIMIT 5
;

"""
    cur.execute(query)
    result = cur.fetchall()

result

[(2, 'The Crenshaw Collector', 2)]

From this query we can see the 'The Crenshaw Collector' has the most emergencies with 2.

### What is the average rating for each exhibition?

In [58]:
with conn.cursor() as cur:
    query = """
    SELECT
        e.exhibition_id,
        e.exhibition_name,
        ROUND(AVG(r.rating_value), 2) AS average_rating
    FROM rating_interaction ri
    JOIN rating r
        USING(rating_id)
    JOIN exhibition e
        USING(exhibition_id)
    GROUP BY e.exhibition_id, e.exhibition_name
    ORDER BY average_rating DESC
;

"""
    cur.execute(query)
    result = cur.fetchall()

result

[(3, 'Cetacean Sensations', Decimal('2.83')),
 (1, 'Adaptation', Decimal('1.93')),
 (0, 'Measureless to Man', Decimal('1.92')),
 (2, 'The Crenshaw Collector', Decimal('1.43')),
 (4, 'Our Polluted World', Decimal('1.22'))]

We can see that the highest rated exhibition is Cetacean Sensation at 2.83.

### Are positive ratings more frequent before or after 1pm?

In [59]:
with conn.cursor() as cur:
    query = """
    SELECT
        CASE
            WHEN EXTRACT(HOUR FROM ri.event_at) < 13 THEN 'Before 1pm'
            ELSE 'After 1pm'
        END AS time_period,
        COUNT(*) AS positive_rating_count
    FROM rating_interaction ri
    JOIN rating r
        USING(rating_id)
    WHERE rating_value >= 3
    GROUP BY time_period
    ORDER BY positive_rating_count DESC
    ;

"""
    cur.execute(query)
    result = cur.fetchall()

result

[('After 1pm', 344), ('Before 1pm', 313)]

From this query we can see that positive ratings are more common before 1pm.

### Do Zoology exhibitions get better ratings than other types?

In [62]:
with conn.cursor() as cur:
    query = """
    SELECT
        CASE
            WHEN d.department_name = 'Zoology' THEN 'Zoology'
            ELSE 'Other departments'
        END AS exhibition_type,
        ROUND(AVG(r.rating_value), 2) AS average_rating,
        COUNT(*) AS rating_count
    FROM rating_interaction ri
    JOIN rating r
        USING(rating_id)
    JOIN exhibition e
        USING(exhibition_id)
    JOIN department d
        USING(department_id)
    GROUP BY exhibition_type
;

"""
    cur.execute(query)
    result = cur.fetchall()

result

[('Other departments', Decimal('1.64'), 1207),
 ('Zoology', Decimal('2.20'), 857)]

The Zoology department does have better ratings when compared to the rest of the departments as a whole.